In [1]:
%pip -q install datasets pandas numpy matplotlib scikit-learn scipy datasketch transformers sentence-transformers pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 3.8 MB/s eta 0:00:00


In [2]:
%%writefile analyze_mf3qa.py
#!/usr/bin/env python3
"""
Analysis of MF3QA (gaokerena/MF3QA) and its raw crawl (gaokerena/MF3QA_uncleaned).

Columns expected (case-insensitive): Question, Answer, Source. The cleaned set has
train/dev/test splits; the uncleaned set has a single split.

Stages (run all by default, or choose with --stages):
  stats          lengths per split/source, short/referral-only answers, train-vs-test shift (KS test)
  tokens         token counts per split (checks the "~3M tokens" claim); needs transformers
  funnel         uncleaned -> cleaned survival by source / answer length / question length
  align          question-answer alignment score + shuffled baseline; exports lowest-scoring pairs
  style          answer-style proxies (stock LLM phrases, ZWNJ use, colloquial forms, position effect)
  pii            URLs, phones, emails, social handles, "Dr. <name>" in questions/answers (counts + row ids only)
  sensitive      keyword screens: self-harm, first-person minors, infant/child dosing, sexual health, pregnancy
  dedup          exact / near duplicates within and across splits; test->train nearest-neighbour similarity;
                 most frequent answers
  topics         TF-IDF + NMF topics and topic share per split / source
  contamination  n-gram overlap of MF3QA items with a corpus (e.g. gaokerena/medical_corpus), needs --corpus
  audit          stratified random sample (CSV) for manual annotation

Outputs go to --out (default ./mf3qa_analysis): CSV tables, summary.json, PNG figures and LaTeX tables.

Install:   pip install datasets pandas numpy matplotlib scikit-learn scipy datasketch transformers sentence-transformers
Run:       python analyze_mf3qa.py
Smoke test: python analyze_mf3qa.py --limit 2000 --stages stats,funnel,style,pii,sensitive,dedup --align-model tfidf

NOTE: the style/sensitive/PII screens are heuristics (regex/keyword proxies). Treat the counts as
flags for manual review, not as ground truth; the `audit` stage produces the sample for that.
"""
import argparse
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Text helpers (same normalisation as analyze_corpus.py)
# ----------------------------------------------------------------------------
DIACRITICS_RE = re.compile(r"[\u064B-\u065F\u0670]")
PUNCT_RE = re.compile(r"[^\w\s]|_")
AR_YEH, AR_KAF, AR_ALEF_MAKSURA = "\u064a", "\u0643", "\u0649"
ZWNJ = "\u200c"
MASK64 = 0xFFFFFFFFFFFFFFFF

_TRANS = {AR_YEH: "\u06cc", AR_KAF: "\u06a9", AR_ALEF_MAKSURA: "\u06cc",
          ZWNJ: " ", "\u200f": " ", "\u200e": " "}
for i in range(10):
    _TRANS[0x06F0 + i] = str(i)
    _TRANS[0x0660 + i] = str(i)
_TRANS = str.maketrans({(k if isinstance(k, str) else chr(k)): v for k, v in _TRANS.items()})
_DIGITS_ONLY = str.maketrans({chr(0x06F0 + i): str(i) for i in range(10)} |
                             {chr(0x0660 + i): str(i) for i in range(10)})


def norm(text: str) -> str:
    t = DIACRITICS_RE.sub("", text).translate(_TRANS).lower()
    t = PUNCT_RE.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()


def md5(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()


def latex_escape(s) -> str:
    return (str(s).replace("\\", "\\textbackslash{}").replace("_", "\\_").replace("&", "\\&")
            .replace("%", "\\%").replace("#", "\\#"))


def to_latex(df, path, caption, label, float_fmt="{:,.1f}"):
    cols = list(df.columns)
    lines = ["\\begin{table}[h]", f"\\caption{{{caption}}}\\label{{{label}}}",
             "\\begin{tabular}{@{}l" + "r" * (len(cols) - 1) + "@{}}", "\\toprule",
             " & ".join(latex_escape(c) for c in cols) + " \\\\", "\\midrule"]
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if isinstance(v, (int, np.integer)):
                cells.append(f"{v:,}")
            elif isinstance(v, (float, np.floating)):
                cells.append("--" if np.isnan(v) else float_fmt.format(v))
            else:
                cells.append(latex_escape(v))
        lines.append(" & ".join(cells) + " \\\\")
    lines += ["\\botrule", "\\end{tabular}", "\\end{table}"]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def save(df, out, name, **kw):
    df.to_csv(os.path.join(out, name), encoding="utf-8-sig", **kw)


# ----------------------------------------------------------------------------
# Loading
# ----------------------------------------------------------------------------
def load_table(spec: str) -> pd.DataFrame:
    """spec: local .parquet/.csv/.jsonl/.json path, or a Hugging Face dataset id (all splits)."""
    ext = os.path.splitext(spec)[1].lower()
    if os.path.exists(spec):
        if ext == ".parquet":
            df = pd.read_parquet(spec)
        elif ext == ".csv":
            df = pd.read_csv(spec)
        elif ext in (".jsonl", ".json"):
            df = pd.read_json(spec, lines=(ext == ".jsonl"))
        else:
            raise ValueError(f"unsupported file type: {spec}")
        if "__split" not in df.columns:
            df["__split"] = "all"
        return df
    from datasets import load_dataset
    ds = load_dataset(spec)
    frames = []
    for split, d in ds.items():
        f = d.to_pandas()
        f["__split"] = split
        frames.append(f)
    return pd.concat(frames, ignore_index=True)


def prep(df: pd.DataFrame, name: str, limit=None) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c if c.startswith("__") else c.lower() for c in df.columns]
    assert {"question", "answer"} <= set(df.columns), f"need Question/Answer columns, got {list(df.columns)}"
    if limit:
        df = df.groupby("__split", group_keys=False).head(limit).copy()
    if "source" not in df.columns:
        df["source"] = "NA"
    df["source"] = df["source"].fillna("NA").astype(str)
    df["question"] = df["question"].fillna("").astype(str)
    df["answer"] = df["answer"].fillna("").astype(str)
    df["dataset"] = name
    df["split"] = df["__split"].astype(str)
    df["id"] = df.groupby("split").cumcount()
    df["q_norm"] = df["question"].map(norm)
    df["a_norm"] = df["answer"].map(norm)
    df["q_words"] = df["q_norm"].map(lambda s: s.count(" ") + 1 if s else 0)
    df["a_words"] = df["a_norm"].map(lambda s: s.count(" ") + 1 if s else 0)
    df["q_chars"] = df["question"].str.len()
    df["a_chars"] = df["answer"].str.len()
    return df[["dataset", "split", "id", "source", "question", "answer", "q_norm", "a_norm",
               "q_words", "a_words", "q_chars", "a_chars"]]


# ----------------------------------------------------------------------------
# Stages
# ----------------------------------------------------------------------------
def desc(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return dict(mean=np.nan, median=np.nan, p5=np.nan, p95=np.nan)
    return dict(mean=x.mean(), median=np.median(x), p5=np.percentile(x, 5), p95=np.percentile(x, 95))


REFERRAL_RE = re.compile(r"(مراجعه|معاینه|ویزیت|متخصص|پزشک)")


def stage_stats(C, out, summary):
    allp = C["all"]
    rows = []
    for (ds, sp), g in allp.groupby(["dataset", "split"]):
        qd, ad = desc(g.q_words), desc(g.a_words)
        short = g.a_words < 30
        referral_only = (g.a_words <= 15) & g.a_norm.map(lambda s: bool(REFERRAL_RE.search(s)))
        rows.append({"dataset": ds, "split": sp, "rows": len(g),
                     "q_words_median": qd["median"], "q_words_mean": qd["mean"],
                     "a_words_median": ad["median"], "a_words_mean": ad["mean"],
                     "a_words_p5": ad["p5"], "a_words_p95": ad["p95"],
                     "answers_lt30_words_%": 100 * short.mean(),
                     "short_referral_only_%": 100 * referral_only.mean(),
                     "empty_q_%": 100 * (g.q_words == 0).mean(), "empty_a_%": 100 * (g.a_words == 0).mean()})
    t = pd.DataFrame(rows)
    save(t, out, "stats_per_split.csv", index=False)
    src = (allp.groupby(["dataset", "split", "source"])
           .agg(rows=("id", "size"), q_words_median=("q_words", "median"), a_words_median=("a_words", "median"),
                a_words_mean=("a_words", "mean")).reset_index())
    save(src, out, "stats_per_split_source.csv", index=False)
    cl = t[t.dataset == "cleaned"]
    if len(cl):
        to_latex(cl[["split", "rows", "q_words_median", "a_words_median", "a_words_mean",
                     "answers_lt30_words_%"]].rename(columns={
                         "split": "Split", "rows": "Rows", "q_words_median": "Q median (words)",
                         "a_words_median": "A median (words)", "a_words_mean": "A mean (words)",
                         "answers_lt30_words_%": "A < 30 words (\\%)"}),
                 os.path.join(out, "table_split_stats.tex"), "MF3QA split statistics.", "tab:mf3qa_splits")
    res = {"per_split": t.to_dict("records")}
    try:
        from scipy.stats import ks_2samp
        c = C["clean"]
        tr, te = c[c.split == "train"], c[c.split == "test"]
        if len(tr) and len(te):
            res["ks_answer_words_train_vs_test"] = dict(zip(("statistic", "pvalue"),
                                                            map(float, ks_2samp(tr.a_words, te.a_words))))
            res["ks_question_words_train_vs_test"] = dict(zip(("statistic", "pvalue"),
                                                              map(float, ks_2samp(tr.q_words, te.q_words))))
    except ImportError:
        log("!! scipy not installed; skipping KS tests")
    summary["stats"] = res
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for (ds, sp), g in allp.groupby(["dataset", "split"]):
        ax[0].hist(np.log10(g.q_words.clip(lower=1)), bins=50, histtype="step", density=True, label=f"{ds}/{sp}")
        ax[1].hist(np.log10(g.a_words.clip(lower=1)), bins=50, histtype="step", density=True, label=f"{ds}/{sp}")
    ax[0].set_xlabel("log10(words per question)")
    ax[1].set_xlabel("log10(words per answer)")
    ax[0].set_ylabel("density")
    ax[1].legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(out, "fig_mf3qa_lengths.png"), dpi=200)
    plt.close()
    log("stats done")


def stage_tokens(C, out, summary, tokenizers, batch=512):
    from transformers import AutoTokenizer
    c = C["clean"]
    res, rows = {}, []
    for name in tokenizers:
        try:
            tok = AutoTokenizer.from_pretrained(name)
            tok.model_max_length = 10 ** 9
        except Exception as e:
            log(f"!! could not load tokenizer {name}: {e}")
            continue
        log(f"tokenising with {name}")

        def count(texts):
            out_ = []
            for i in range(0, len(texts), batch):
                out_.extend(len(x) for x in tok(texts[i:i + batch], add_special_tokens=False)["input_ids"])
            return np.array(out_)
        qt, at = count(c.question.tolist()), count(c.answer.tolist())
        tmp = c.assign(q_tok=qt, a_tok=at)
        for sp, g in tmp.groupby("split"):
            rows.append({"tokenizer": name, "split": sp, "rows": len(g), "q_tokens": int(g.q_tok.sum()),
                         "a_tokens": int(g.a_tok.sum()), "total_tokens": int(g.q_tok.sum() + g.a_tok.sum()),
                         "a_tokens_median": float(g.a_tok.median()),
                         "answers_lt50_tokens_%": float(100 * (g.a_tok < 50).mean())})
        res[name] = {"fertility_tokens_per_word_A": float(at.sum() / max(1, c.a_words.sum()))}
    if rows:
        t = pd.DataFrame(rows)
        save(t, out, "tokens_per_split.csv", index=False)
        res["table"] = rows
    summary["tokens"] = res
    log("tokens done")


def stage_funnel(C, out, summary):
    c, u = C.get("clean"), C.get("unclean")
    if c is None or u is None:
        log("!! funnel needs both cleaned and uncleaned sets; skipping")
        return
    ckeys = set(c.q_norm.map(md5))
    u = u.copy()
    u["qkey"] = u.q_norm.map(md5)
    u["survived"] = u.qkey.isin(ckeys)
    u["is_empty"] = (u.q_words == 0) | (u.a_words == 0)
    abins = [-1, 10, 25, 50, 100, 200, 1e9]
    alab = ["<=10", "11-25", "26-50", "51-100", "101-200", ">200"]
    u["a_len_bin"] = pd.cut(u.a_words, abins, labels=alab)
    u["q_len_bin"] = pd.cut(u.q_words, [-1, 10, 25, 50, 100, 1e9], labels=["<=10", "11-25", "26-50", "51-100", ">100"])
    by_src = u.groupby("source").agg(raw_rows=("id", "size"), kept=("survived", "sum"))
    by_src["kept_%"] = 100 * by_src.kept / by_src.raw_rows
    save(by_src, out, "funnel_by_source.csv")
    by_a = u.groupby("a_len_bin", observed=False).agg(raw_rows=("id", "size"), kept=("survived", "sum"))
    by_a["kept_%"] = 100 * by_a.kept / by_a.raw_rows
    save(by_a, out, "funnel_by_answer_length.csv")
    by_q = u.groupby("q_len_bin", observed=False).agg(raw_rows=("id", "size"), kept=("survived", "sum"))
    by_q["kept_%"] = 100 * by_q.kept / by_q.raw_rows
    save(by_q, out, "funnel_by_question_length.csv")
    ukeys = set(u.qkey)
    c = c.copy()
    c["in_raw"] = c.q_norm.map(md5).isin(ukeys)
    unmatched = c.groupby(["split", "source"]).agg(rows=("id", "size"), not_in_raw=("in_raw", lambda x: int((~x).sum())))
    save(unmatched, out, "cleaned_rows_not_in_uncleaned.csv")
    tb = by_src.reset_index().sort_values("raw_rows", ascending=False)
    to_latex(tb[["source", "raw_rows", "kept", "kept_%"]].rename(columns={
        "source": "Source", "raw_rows": "Raw", "kept": "Kept", "kept_%": "Kept (\\%)"}),
        os.path.join(out, "table_funnel.tex"), "Survival of raw MF3QA records through cleaning, by source "
        "(matched on normalised question text).", "tab:mf3qa_funnel")
    summary["funnel"] = {
        "raw_rows": int(len(u)), "cleaned_rows": int(len(c)),
        "raw_empty_question_or_answer": int(u.is_empty.sum()),
        "raw_unique_questions": int(u.qkey.nunique()),
        "raw_rows_matched_in_cleaned": int(u.survived.sum()),
        "removed_%_vs_cleaned_size": float(100 * (1 - len(c) / len(u))),
        "cleaned_rows_with_no_raw_match": int((~c.in_raw).sum()),
        "cleaned_rows_with_no_raw_match_by_split": {k: int(v) for k, v in
                                                    c[~c.in_raw].groupby("split").size().items()}}
    log("funnel done")


def _embed_scores(q, a, model_name, seed):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(a))
    if model_name == "tfidf":
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(token_pattern=r"\S{2,}", min_df=2, sublinear_tf=True)
        vec.fit(list(q) + list(a))
        Q, A = vec.transform(q), vec.transform(a)
        true = np.asarray(Q.multiply(A).sum(1)).ravel()
        shuf = np.asarray(Q.multiply(A[perm]).sum(1)).ravel()
        return true, shuf, "tfidf (lexical proxy)"
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(model_name)
    log(f"embedding {len(q):,} questions + {len(a):,} answers on device: {m.device} "
        f"({'fast' if str(m.device).startswith('cuda') else 'CPU: this can take 5-15 min; use --align-limit to subsample'})")
    eq = m.encode([x[:600] for x in q], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    ea = m.encode([x[:600] for x in a], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    return (eq * ea).sum(1), (eq * ea[perm]).sum(1), model_name


def stage_align(C, out, summary, model_name, seed, limit=None):
    c = C["clean"].copy()
    if limit and len(c) > limit:
        c = c.sample(limit, random_state=seed).sort_values(["split", "id"]).reset_index(drop=True)
        log(f"align: using a random sample of {limit:,} cleaned rows")
    try:
        true, shuf, used = _embed_scores(c.question.tolist(), c.answer.tolist(), model_name, seed)
    except Exception as e:
        log(f"!! embedding model failed ({e}); falling back to tfidf proxy")
        true, shuf, used = _embed_scores(c.q_norm.tolist(), c.a_norm.tolist(), "tfidf", seed)
    c["sim"] = true
    c["sim_shuffled"] = shuf
    thr_med, thr_p95 = float(np.median(shuf)), float(np.percentile(shuf, 95))
    c["likely_mismatch"] = c.sim < thr_med
    c["weak_alignment"] = c.sim < thr_p95
    per = c.groupby(["split", "source"]).agg(rows=("id", "size"), sim_mean=("sim", "mean"),
                                             likely_mismatch=("likely_mismatch", "sum"),
                                             weak_alignment=("weak_alignment", "sum"))
    per["likely_mismatch_%"] = 100 * per.likely_mismatch / per.rows
    per["weak_alignment_%"] = 100 * per.weak_alignment / per.rows
    save(per, out, "alignment_per_split_source.csv")
    save(c[["split", "id", "source", "sim", "sim_shuffled"]], out, "alignment_scores.csv", index=False)
    save(c.nsmallest(200, "sim")[["split", "id", "source", "sim", "question", "answer"]], out,
         "alignment_lowest200_for_review.csv", index=False)
    summary["align"] = {
        "method": used, "sim_true": desc(true), "sim_shuffled": desc(shuf),
        "threshold_shuffled_median": thr_med, "threshold_shuffled_p95": thr_p95,
        "likely_mismatch_rows": int(c.likely_mismatch.sum()), "likely_mismatch_%": float(100 * c.likely_mismatch.mean()),
        "weak_alignment_%": float(100 * c.weak_alignment.mean()),
        "note": "likely_mismatch = true-pair similarity below the median of random-pair similarity; "
                "confirm on the manual audit sample"}
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 4))
    plt.hist(shuf, bins=60, alpha=0.6, density=True, label="random pairs")
    plt.hist(true, bins=60, alpha=0.6, density=True, label="true pairs")
    plt.xlabel("question-answer similarity")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out, "fig_mf3qa_alignment.png"), dpi=200)
    plt.close()
    log(f"align done ({used})")


STOCK_PHRASES = ["بر اساس توضیحاتی که", "بر اساس اطلاعاتی که", "با توجه به اطلاعاتی که", "با توجه به توضیحات",
                 "توصیه می‌شود", "مهم است که", "نیازمند بررسی", "می‌تواند نشان‌دهنده", "ممکن است ناشی از",
                 "در نهایت", "لازم است که", "به عنوان یک", "در پاسخ به سوال شما", "قبل از هر اقدامی"]
STOCK_NORM = [norm(p) for p in STOCK_PHRASES]
COLLOQ_RE = re.compile(r"\b(میشه|میتونه|نمیشه|بشه|کنین|بدین|نکنین|دارین|میخوره|بخورین|میتونین|نمیتونه|باشه)\b")
GREETING_RE = re.compile(r"^(سلام|باسلام|با سلام|درود|وقت بخیر|سلام و)")


def stage_style(C, out, summary):
    c = C["clean"].copy()
    a_norm, a_raw = c.a_norm, c.answer
    c["stock_hits"] = a_norm.map(lambda s: sum(p in s for p in STOCK_NORM))
    c["colloq"] = a_norm.map(lambda s: len(COLLOQ_RE.findall(s)))
    c["zwnj_per_100w"] = [100 * r.count(ZWNJ) / max(1, w) for r, w in zip(a_raw, c.a_words)]
    c["greeting"] = a_norm.map(lambda s: bool(GREETING_RE.match(s)))
    c["llm_like"] = (c.stock_hits >= 1) | ((c.zwnj_per_100w >= 3) & (c.colloq == 0) & (c.a_words >= 60))
    per = c.groupby(["split", "source"]).agg(rows=("id", "size"), llm_like=("llm_like", "mean"),
                                             stock_phrase=("stock_hits", lambda x: (x > 0).mean()),
                                             colloquial=("colloq", lambda x: (x > 0).mean()),
                                             greeting=("greeting", "mean"), zwnj_per_100w=("zwnj_per_100w", "mean"))
    for k in ("llm_like", "stock_phrase", "colloquial", "greeting"):
        per[k] = 100 * per[k]
    save(per, out, "style_per_split_source.csv")
    tr = c[c.split == "train"].sort_values("id")
    dec = pd.qcut(tr.id.rank(method="first"), 10, labels=False) if len(tr) >= 10 else pd.Series(0, index=tr.index)
    pos = tr.assign(decile=dec).groupby("decile").llm_like.mean() * 100
    save(pos.rename("llm_like_%").to_frame(), out, "style_llm_like_by_train_position_decile.csv")
    summary["style"] = {
        "llm_like_%_overall": float(100 * c.llm_like.mean()),
        "stock_phrase_%": float(100 * (c.stock_hits > 0).mean()),
        "colloquial_forms_%": float(100 * (c.colloq > 0).mean()),
        "note": "heuristic proxy for LLM-style text (stock phrases, or ZWNJ-rich formal register without colloquial forms); "
                "validate against the manual audit"}
    log("style done")


# ---- PII ----
URL_RE = re.compile(r"(?:https?://|www\.)\S+|\b[\w-]+\.(?:com|ir|org|net|me)\b\S*", re.I)
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
MOBILE_RE = re.compile(r"(?<!\d)(?:\+98|0098|0)?9\d{9}(?!\d)")
LANDLINE_RE = re.compile(r"(?<!\d)0(?!9)\d{2}[- ]?\d{7,8}(?!\d)")
HANDLE_RE = re.compile(r"(?:t\.me|telegram\.me|instagram\.com)/\S+|(?<![\w.])@\w{3,}", re.I)
DR_NAME_RE = re.compile(r"دکتر\s+([^\s\d.,،؟?!:؛]+)")
DR_STOP = {"عزیز", "محترم", "گرامی", "و", "من", "یا", "که", "را", "برای", "رفتم", "گفت", "گفتن", "بهم", "جان",
           "های", "ها", "اطفال", "زنان", "پوست", "چشم", "قلب", "مغز", "داخلی", "عمومی", "متخصص", "خانم", "آقای"}


def _pii_flags(text):
    t = text.translate(_DIGITS_ONLY)
    names = [m for m in DR_NAME_RE.findall(t) if m not in DR_STOP and len(m) > 2]
    return {"url": bool(URL_RE.search(t)), "email": bool(EMAIL_RE.search(t)),
            "mobile": bool(MOBILE_RE.search(t)), "landline": bool(LANDLINE_RE.search(t)),
            "social_handle": bool(HANDLE_RE.search(t)), "dr_name_approx": bool(names)}


def stage_pii(C, out, summary):
    frames = [x for x in (C.get("clean"), C.get("unclean")) if x is not None]
    rows, ids = [], []
    for df in frames:
        for field in ("question", "answer"):
            fl = pd.DataFrame([_pii_flags(t) for t in df[field]], index=df.index)
            g = pd.concat([df[["dataset", "split", "id", "source"]], fl], axis=1)
            for (ds, sp), gg in g.groupby(["dataset", "split"]):
                r = {"dataset": ds, "split": sp, "field": field, "rows": len(gg)}
                for k in fl.columns:
                    r[k + "_rows"] = int(gg[k].sum())
                    r[k + "_%"] = float(100 * gg[k].mean())
                rows.append(r)
            hit = g[fl.any(axis=1)][["dataset", "split", "id", "source"]].copy()
            hit["field"] = field
            for k in fl.columns:
                hit[k] = g.loc[hit.index, k]
            ids.append(hit)
    t = pd.DataFrame(rows)
    save(t, out, "pii_counts.csv", index=False)
    if ids:
        save(pd.concat(ids), out, "pii_flagged_row_ids.csv", index=False)
    summary["pii"] = {"table": rows, "note": "counts and row ids only (no PII values exported); doctor-name detector is approximate"}
    log("pii done")


SELFHARM_RE = re.compile(r"(خودکشی|خودزنی|خود زنی|بریدن دست|خودم رو بکشم|خودمو بکشم|تیغ)")
FIRST_PERSON_AGE_RE = re.compile(r"(?:من|بنده)\s*(\d{1,2})\s*(?:ساله|سالمه|سالم)|(?<!\d)(\d{1,2})\s*سالمه")
CHILD_WORD_RE = re.compile(r"(نوزاد|شیرخوار|کودک|بچه|ماهه|پسرم|دخترم)")
DOSE_WORD_RE = re.compile(r"(دوز|سی سی|میلی|قطره|شربت|قرص|mg|cc|آمپول|شیاف)")
SEXUAL_RE = re.compile(r"(بکارت|رابطه جنسی|دخول|خودارضایی|خود ارضایی|تناسلی|واژن)")
PREG_RE = re.compile(r"(باردار|حامله|بارداری)")


def stage_sensitive(C, out, summary):
    frames = [x for x in (C.get("clean"), C.get("unclean")) if x is not None]
    rows, flagged = [], []
    for df in frames:
        qn = df.q_norm
        ages = qn.map(lambda s: [int(x) for m in FIRST_PERSON_AGE_RE.findall(s) for x in m if x])
        f = pd.DataFrame({
            "self_harm_kw": qn.map(lambda s: bool(SELFHARM_RE.search(s))),
            "first_person_minor": ages.map(lambda a: any(0 < x < 18 for x in a)),
            "child_dosing": qn.map(lambda s: bool(CHILD_WORD_RE.search(s)) and bool(DOSE_WORD_RE.search(s))),
            "sexual_health": qn.map(lambda s: bool(SEXUAL_RE.search(s))),
            "pregnancy": qn.map(lambda s: bool(PREG_RE.search(s)))}, index=df.index)
        g = pd.concat([df[["dataset", "split", "id", "source"]], f], axis=1)
        for (ds, sp, src), gg in g.groupby(["dataset", "split", "source"]):
            r = {"dataset": ds, "split": sp, "source": src, "rows": len(gg)}
            for k in f.columns:
                r[k + "_rows"] = int(gg[k].sum())
                r[k + "_%"] = float(100 * gg[k].mean())
            rows.append(r)
        flagged.append(g[f["self_harm_kw"] | f["first_person_minor"]][["dataset", "split", "id", "source"] +
                                                                    ["self_harm_kw", "first_person_minor"]])
    t = pd.DataFrame(rows)
    save(t, out, "sensitive_counts.csv", index=False)
    if flagged:
        save(pd.concat(flagged), out, "sensitive_selfharm_or_minor_row_ids.csv", index=False)
    summary["sensitive"] = {
        "note": "keyword/regex screens on questions only; they over- and under-count. Use the row-id list for manual review "
                "and decide on a content warning / filtered release.",
        "totals": t.groupby(["dataset", "split"])[[c for c in t.columns if c.endswith("_rows")] + ["rows"]]
        .sum().reset_index().to_dict("records")}
    log("sensitive done")


def stage_dedup(C, out, summary, neardup=True, threshold=0.8, num_perm=64, seed=0):
    c = C["clean"].copy()
    c["qkey"] = c.q_norm.map(md5)
    c["qakey"] = (c.q_norm + "||" + c.a_norm).map(md5)
    res = {}
    # exact duplicates within each split and leakage across splits
    for key in ("qkey", "qakey"):
        res[f"{key}_dup_within_split_rows"] = {sp: int(g.duplicated(key).sum()) for sp, g in c.groupby("split")}
        keysets = {sp: set(g[key]) for sp, g in c.groupby("split")}
        cross = {}
        for a in keysets:
            for b in keysets:
                if a != b:
                    cross[f"{a}_rows_with_{key}_in_{b}"] = int(c[(c.split == a)][key].isin(keysets[b]).sum())
        res.update(cross)
    # near duplicates (questions) via MinHash LSH
    if neardup:
        try:
            from datasketch import MinHash, MinHashLSH
            lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
            mhs = {}
            for i, text in enumerate(c.q_norm.tolist()):
                w = text.split()
                if len(w) < 5:
                    continue
                m = MinHash(num_perm=num_perm)
                sh = {" ".join(w[k:k + 3]).encode() for k in range(len(w) - 2)}
                if hasattr(m, "update_batch"):
                    m.update_batch(list(sh))
                else:
                    for s in sh:
                        m.update(s)
                mhs[i] = m
                lsh.insert(str(i), m)
            splits = c.split.tolist()
            cross_near = {}
            within = {}
            for i, m in mhs.items():
                for h in lsh.query(m):
                    j = int(h)
                    if j == i:
                        continue
                    k = (splits[i], splits[j])
                    if k[0] == k[1]:
                        within[k[0]] = within.get(k[0], 0) + 1
                    else:
                        cross_near.setdefault(f"{k[0]}_rows_with_near_dup_question_in_{k[1]}", set()).add(i)
            res["near_dup_question_pairs_within_split"] = within
            res.update({k: len(v) for k, v in cross_near.items()})
            res["near_dup_params"] = f"3-word shingles on questions (>=5 words), Jaccard>={threshold}, {num_perm} perms"
        except ImportError:
            log("!! datasketch not installed; skipping near-duplicates (pip install datasketch)")
    # nearest neighbour similarity to train (word uni+bigram tf-idf cosine)
    from sklearn.feature_extraction.text import TfidfVectorizer
    tr = c[c.split == "train"]
    if len(tr):
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True, token_pattern=r"\S+")
        Xtr = vec.fit_transform(tr.q_norm)
        nn = {}
        for sp in [s for s in c.split.unique() if s != "train"]:
            g = c[c.split == sp]
            X = vec.transform(g.q_norm)
            best = np.zeros(X.shape[0])
            for s0 in range(0, X.shape[0], 500):
                sim = (X[s0:s0 + 500] @ Xtr.T).toarray()
                best[s0:s0 + 500] = sim.max(1) if sim.size else 0
            nn[sp] = {"median": float(np.median(best)), "p95": float(np.percentile(best, 95)),
                      "share_ge_0.8_%": float(100 * (best >= 0.8).mean()),
                      "share_ge_0.9_%": float(100 * (best >= 0.9).mean())}
            save(pd.DataFrame({"id": g.id.values, "max_cos_to_train": best}), out, f"nn_to_train_{sp}.csv", index=False)
        res["nearest_train_question_similarity"] = nn
    # most frequent answers
    top = []
    for name in ("clean", "unclean"):
        df = C.get(name)
        if df is None:
            continue
        vc = df.a_norm[df.a_norm != ""].value_counts().head(30)
        top.append(pd.DataFrame({"dataset": name, "count": vc.values, "answer": vc.index}))
        res[f"{name}_rows_with_answer_repeated_>=5_times_%"] = float(
            100 * df.a_norm.map(df.a_norm.value_counts()).ge(5).mean())
    if top:
        save(pd.concat(top), out, "most_frequent_answers.csv", index=False)
    summary["dedup"] = res
    log("dedup done")


def stage_topics(C, out, summary, n_topics=12, sample=20000, max_words=300, seed=0):
    from sklearn.decomposition import NMF
    from sklearn.feature_extraction.text import TfidfVectorizer
    c = C["clean"]
    d = c.sample(min(sample, len(c)), random_state=seed)
    texts = (d.q_norm + " " + d.a_norm).map(lambda s: " ".join(s.split()[:max_words]))
    vec = TfidfVectorizer(max_features=20000, min_df=5, max_df=0.4, token_pattern=r"\S{2,}")
    X = vec.fit_transform(texts)
    nmf = NMF(n_components=n_topics, random_state=seed, init="nndsvda", max_iter=300)
    W = nmf.fit_transform(X)
    terms = np.array(vec.get_feature_names_out())
    top = W.argmax(1)
    rows = [{"topic": k, "top_terms": " ".join(terms[comp.argsort()[::-1][:12]]), "docs_%": float(100 * (top == k).mean())}
            for k, comp in enumerate(nmf.components_)]
    save(pd.DataFrame(rows), out, "topics.csv", index=False)
    save(pd.crosstab(d.split.values, top, normalize="index"), out, "topic_share_per_split.csv")
    save(pd.crosstab(d.source.values, top, normalize="index"), out, "topic_share_per_source.csv")
    summary["topics"] = {"n_topics": n_topics, "sample_rows": int(len(d))}
    log("topics done (label the topics manually from topics.csv)")


def ngram_hashes(words, n):
    if len(words) < n:
        return np.empty(0, dtype=np.uint64)
    h = np.fromiter((hash(w) & MASK64 for w in words), dtype=np.uint64, count=len(words))
    m = len(words) - n + 1
    out = np.zeros(m, dtype=np.uint64)
    mult = np.uint64(1000003)
    with np.errstate(over="ignore"):
        for j in range(n):
            out = out * mult + h[j:j + m]
    return out


def stage_contamination(C, out, summary, corpus_spec, n=8, limit=None):
    if not corpus_spec:
        log("!! contamination needs --corpus (e.g. gaokerena/medical_corpus); skipping")
        return
    corpus = load_table(corpus_spec)
    col = "content" if "content" in corpus.columns else [c for c in corpus.columns if not c.startswith("__")][-1]
    if limit:
        corpus = corpus.head(limit)
    c = C["clean"]
    items = [(sp, (q + " " + a).split()) for sp, q, a in zip(c.split, c.q_norm, c.a_norm)]
    grams = [ngram_hashes(w, n) for _, w in items]
    allg = np.unique(np.concatenate([g for g in grams if len(g)])) if any(len(g) for g in grams) else np.empty(0, np.uint64)
    if len(allg) == 0:
        log("!! no evaluable items")
        return
    log(f"scanning {len(corpus):,} corpus docs for {len(allg):,} distinct {n}-grams from {len(items):,} MF3QA items")
    matched = []
    for text in corpus[col].fillna("").astype(str):
        g = ngram_hashes(norm(text).split(), n)
        if len(g) == 0:
            continue
        idx = np.searchsorted(allg, g)
        idx[idx == len(allg)] = 0
        hit = allg[idx] == g
        if hit.any():
            matched.append(np.unique(g[hit]))
    matched = np.unique(np.concatenate(matched)) if matched else np.empty(0, np.uint64)
    frac = np.array([np.isin(g, matched).mean() if len(g) else np.nan for g in grams])
    splits = np.array([s for s, _ in items])
    rows = []
    for sp in sorted(set(splits)):
        f = frac[splits == sp]
        ok = f[~np.isnan(f)]
        rows.append({"split": sp, "items": int(len(f)), "evaluable": int(len(ok)),
                     f"any_{n}gram_match_%": float(100 * (ok > 0).mean()) if len(ok) else np.nan,
                     "gt_50%_matched_%": float(100 * (ok > 0.5).mean()) if len(ok) else np.nan})
    save(pd.DataFrame(rows), out, "contamination_mf3qa_vs_corpus.csv", index=False)
    summary["contamination"] = rows
    log("contamination done")


def stage_audit(C, out, summary, n=200, seed=0):
    c = C["clean"]
    parts = []
    g = c.groupby(["split", "source"])
    for _, gg in g:
        k = max(1, int(round(n * len(gg) / len(c))))
        parts.append(gg.sample(min(k, len(gg)), random_state=seed))
    s = pd.concat(parts).sample(frac=1, random_state=seed).head(n)
    s = s[["split", "id", "source", "question", "answer"]].copy()
    for col in ("aligned_0_1", "factually_ok_0_1_2", "llm_generated_0_1", "contains_pii_0_1", "unsafe_0_1", "notes"):
        s[col] = ""
    save(s, out, "audit_sample_for_annotation.csv", index=False)
    summary["audit"] = {"rows": int(len(s)),
                        "protocol": "two annotators, report Cohen's kappa for aligned / llm_generated / unsafe"}
    log("audit sample written")


# ----------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------
ALL_STAGES = ["stats", "tokens", "funnel", "align", "style", "pii", "sensitive", "dedup", "topics", "contamination", "audit"]


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--cleaned", default="gaokerena/MF3QA", help="HF id or local file")
    ap.add_argument("--uncleaned", default="gaokerena/MF3QA_uncleaned", help="HF id, local file, or 'none'")
    ap.add_argument("--out", default="mf3qa_analysis")
    ap.add_argument("--limit", type=int, default=None, help="max rows per split (smoke tests)")
    ap.add_argument("--stages", default=",".join(ALL_STAGES))
    ap.add_argument("--tokenizers", default="CohereLabs/aya-expanse-8b,Qwen/Qwen2.5-7B",
                    help="aya-expanse is gated: accept its licence and run `huggingface-cli login`")
    ap.add_argument("--align-model", default="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                    help="sentence-transformers model id, or 'tfidf' for a lexical proxy")
    ap.add_argument("--align-limit", type=int, default=None,
                    help="embed only a random sample of N cleaned rows in the align stage (speed on CPU)")
    ap.add_argument("--corpus", default=None, help="corpus for the contamination stage, e.g. gaokerena/medical_corpus")
    ap.add_argument("--corpus-limit", type=int, default=None)
    ap.add_argument("--ngram", type=int, default=8)
    ap.add_argument("--no-neardup", action="store_true")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    stages = [s.strip() for s in args.stages.split(",") if s.strip()]
    C = {}
    log(f"loading {args.cleaned}")
    C["clean"] = prep(load_table(args.cleaned), "cleaned", args.limit)
    log(f"cleaned: {len(C['clean']):,} rows, splits {sorted(C['clean'].split.unique())}")
    if args.uncleaned and args.uncleaned.lower() != "none":
        try:
            log(f"loading {args.uncleaned}")
            C["unclean"] = prep(load_table(args.uncleaned), "uncleaned", args.limit)
            log(f"uncleaned: {len(C['unclean']):,} rows")
        except Exception as e:
            log(f"!! could not load uncleaned set: {e}")
    C["all"] = pd.concat([v for k, v in C.items() if k in ("clean", "unclean")], ignore_index=True)
    summary = {"rows": {k: int(len(v)) for k, v in C.items() if k != "all"}}

    if "stats" in stages:
        stage_stats(C, args.out, summary)
    if "tokens" in stages:
        stage_tokens(C, args.out, summary, [t for t in args.tokenizers.split(",") if t])
    if "funnel" in stages:
        stage_funnel(C, args.out, summary)
    if "align" in stages:
        stage_align(C, args.out, summary, args.align_model, args.seed, args.align_limit)
    if "style" in stages:
        stage_style(C, args.out, summary)
    if "pii" in stages:
        stage_pii(C, args.out, summary)
    if "sensitive" in stages:
        stage_sensitive(C, args.out, summary)
    if "dedup" in stages:
        stage_dedup(C, args.out, summary, neardup=not args.no_neardup, seed=args.seed)
    if "topics" in stages:
        stage_topics(C, args.out, summary, seed=args.seed)
    if "contamination" in stages:
        stage_contamination(C, args.out, summary, args.corpus, n=args.ngram, limit=args.corpus_limit)
    if "audit" in stages:
        stage_audit(C, args.out, summary, seed=args.seed)

    with open(os.path.join(args.out, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2, default=str)
    log(f"all done -> {args.out}/summary.json")


if __name__ == "__main__":
    main()

Writing analyze_mf3qa.py


In [3]:
!python analyze_mf3qa.py --limit 2000 --stages stats,funnel,style,pii,sensitive,dedup --align-model tfidf --out mf3qa_smoke_test


[23:16:39] loading gaokerena/MF3QA
README.md: 100% 1.06k/1.06k [00:00<00:00, 589kB/s]

data/train-00000-of-00001.parquet: downloading bytes:  15% 1.70M/11.3M [00:00<00:03, 2.88MB/s, 5.18kB/s  ]
data/train-00000-of-00001.parquet: downloading bytes: 100% 11.2M/11.2M [00:00<00:00, 12.7MB/s, 1.10MB/s  ]
data/train-00000-of-00001.parquet: reconstructing file: 100% 11.3M/11.3M [00:00<00:00, 12.8MB/s, 1.10MB/s  ]

data/dev-00000-of-00001.parquet: downloading bytes:  16% 180k/1.10M [00:00<00:02, 339kB/s]
data/dev-00000-of-00001.parquet: downloading bytes: 100% 1.09M/1.09M [00:00<00:00, 1.85MB/s,  108kB/s  ]
data/dev-00000-of-00001.parquet: reconstructing file: 100% 1.10M/1.10M [00:00<00:00, 1.86MB/s,  108kB/s  ]

data/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/796k [00:00<?, ?B/s]
data/test-00000-of-00001.parquet: downloading bytes: 100% 793k/793k [00:00<00:00, 1.73MB/s, 78.5kB/s  ]
data/test-00000-of-00001.parquet: reconstructing file: 100% 796k/796k [00:00<00:00, 1.74MB/s, 78.

In [4]:
!python analyze_mf3qa.py \
  --stages stats,tokens,funnel,align,style,pii,sensitive,dedup,topics,contamination,audit \
  --corpus gaokerena/medical_corpus \
  --out mf3qa_analysis


[23:17:03] loading gaokerena/MF3QA
[23:17:11] cleaned: 24,000 rows, splits ['dev', 'test', 'train']
[23:17:11] loading gaokerena/MF3QA_uncleaned
[23:17:32] uncleaned: 186,029 rows
[23:17:35] stats done
[23:17:49] !! could not load tokenizer CohereLabs/aya-expanse-8b: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/CohereLabs/aya-expanse-8b.
401 Client Error. (Request ID: Root=1-6acac79d-017a80d70d71bd036d53190b;e0818b94-af52-4abe-94bd-53de4353002d)

Cannot access gated repo for url https://huggingface.co/CohereLabs/aya-expanse-8b/resolve/5062468bf9bc0c6035fd64e06274333ec127d980/config.json.
Access to model CohereLabs/aya-expanse-8b is restricted. You must have access to it and be authenticated to access it. Please log in.
config.json: 100% 686/686 [00:00<00:00, 4.02MB/s]
tokenizer_config.json: 100% 7.23k/7.23k [00:00<00:00, 18.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 77.4MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 107MB/s]
t

In [5]:
import json
from pathlib import Path
import pandas as pd

out = Path("mf3qa_analysis")
with open(out / "summary.json", encoding="utf-8") as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2, ensure_ascii=False))

for filename in [
    "stats_per_split.csv",
    "stats_per_split_source.csv",
    "funnel_by_source.csv",
    "alignment_per_split_source.csv",
    "style_per_split_source.csv",
    "pii_counts.csv",
    "sensitive_counts.csv",
    "contamination_mf3qa_vs_corpus.csv",
    "topics.csv",
]:
    path = out / filename
    if path.exists():
        print("\n###", filename)
        display(pd.read_csv(path).head(20))


{
  "rows": {
    "clean": 24000,
    "unclean": 186029
  },
  "stats": {
    "per_split": [
      {
        "dataset": "cleaned",
        "split": "dev",
        "rows": 2000,
        "q_words_median": 45.0,
        "q_words_mean": 53.1325,
        "a_words_median": 77.0,
        "a_words_mean": 97.761,
        "a_words_p5": 44.0,
        "a_words_p95": 250.24999999999977,
        "answers_lt30_words_%": 0.0,
        "short_referral_only_%": 0.0,
        "empty_q_%": 0.0,
        "empty_a_%": 0.0
      },
      {
        "dataset": "cleaned",
        "split": "test",
        "rows": 2000,
        "q_words_median": 11.0,
        "q_words_mean": 29.3145,
        "a_words_median": 51.0,
        "a_words_mean": 62.151,
        "a_words_p5": 9.0,
        "a_words_p95": 152.0,
        "answers_lt30_words_%": 27.900000000000002,
        "short_referral_only_%": 0.35000000000000003,
        "empty_q_%": 0.0,
        "empty_a_%": 0.0
      },
      {
        "dataset": "cleaned",
        "spli

,dataset,split,rows,q_words_median,q_words_mean,a_words_median,a_words_mean,a_words_p5,a_words_p95,answers_lt30_words_%,short_referral_only_%,empty_q_%,empty_a_%
0,cleaned,dev,2000,45.0,53.132500,77.0,97.761000,44.0,250.25,0.000000,0.000000,0.000000,0.000000
1,cleaned,test,2000,11.0,29.314500,51.0,62.151000,9.0,152.00,27.900000,0.350000,0.000000,0.000000
2,cleaned,train,20000,51.0,55.406500,60.0,75.233450,39.0,159.00,0.015000,0.000000,0.000000,0.000000
3,uncleaned,train,186029,43.0,48.217455,17.0,25.874627,5.0,76.00,74.291105,13.245247,0.002688,0.041929



### stats_per_split_source.csv


,dataset,split,source,rows,q_words_median,a_words_median,a_words_mean
0,cleaned,dev,niniban.com,2000,45.0,77.0,97.761000
1,cleaned,test,doctor-yab.ir,1711,11.0,48.0,58.269433
2,cleaned,test,isovisit.com,88,37.5,37.0,43.704545
3,cleaned,test,kqa,201,9.0,100.0,103.268657
4,cleaned,train,drhast.com,16596,53.0,57.0,70.668776
5,cleaned,train,niniban.com,3404,46.0,76.0,97.488249
6,uncleaned,train,doctor-yab.ir,54539,38.0,14.0,20.564220
7,uncleaned,train,drhast.com,122233,44.0,17.0,25.052580
8,uncleaned,train,isovisit.com,85,38.0,38.0,44.847059
9,uncleaned,train,niniban.com,9172,44.0,48.0,68.231029



### funnel_by_source.csv


...

,dataset,split,source,rows,self_harm_kw_rows,self_harm_kw_%,first_person_minor_rows,first_person_minor_%,child_dosing_rows,child_dosing_%,sexual_health_rows,sexual_health_%,pregnancy_rows,pregnancy_%
0,cleaned,dev,niniban.com,2000,0,0.000000,10,0.500000,158,7.900000,133,6.650000,1118,55.900000
1,cleaned,test,doctor-yab.ir,1711,13,0.759790,23,1.344243,19,1.110462,50,2.922268,49,2.863822
2,cleaned,test,isovisit.com,88,1,1.136364,2,2.272727,3,3.409091,9,10.227273,4,4.545455
3,cleaned,test,kqa,201,0,0.000000,0,0.000000,0,0.000000,3,1.492537,3,1.492537
4,cleaned,train,drhast.com,16596,117,0.704989,132,0.795372,871,5.248253,606,3.651482,1443,8.694866
5,cleaned,train,niniban.com,3404,2,0.058754,22,0.646298,240,7.050529,234,6.874266,1962,57.638073
6,uncleaned,train,doctor-yab.ir,54539,337,0.617906,1165,2.136086,1572,2.882341,5553,10.181705,8650,15.860210
7,uncleaned,train,drhast.com,122233,526,0.430326,884,0.723209,5083,4.158451,6481,5.302169,15333,12.544076
8,uncleaned,train,isovisit.com,85,1,1.176471,2,2.352941,3,3.529412,9,10.588235,4,4.705882
9,uncleaned,train,niniban.com,9172,4,0.043611,66,0.719581,662,7.217619,715,7.795464,5130,55.931095



### contamination_mf3qa_vs_corpus.csv


,split,items,evaluable,any_8gram_match_%,gt_50%_matched_%
0,dev,2000,2000,41.000000,11.700000
1,test,2000,1997,18.477717,2.103155
2,train,20000,20000,14.605000,2.200000



### topics.csv


,topic,top_terms,docs_%
0,0,درمان بیماری شود باشد دارد جراحی تشخیص عمل وجو...,13.690
1,1,تخمک گذاری صد مقاربت جنین کیت پسر توانید روش ه...,2.195
2,2,قرص اورژانسی قاعدگی ضمن پریود عوارض ایجاد مصرف...,1.665
3,3,شیر نوزاد مادر خشک قطره بچه کودک بهش میخوره سی...,5.605
4,4,جنین هفته بارداری سونوگرافی تاریخ سونو سقط شود...,5.655
5,5,زایمان سزارین طبیعی زودرس رحم مادر زنان دهانه ...,3.020
6,6,درد دندان هست معاینه فشار قلب کمر سمت سینه دار...,13.220
7,7,خیلی رو میکنم کنم دارم رابطه ولی یه هست میشه ت...,23.450
8,8,مصرف قرص دارو قطع تجویز استفاده پزشک کنم رو دا...,11.710
9,9,تست پریود روز بتا آزمایش خون تاخیر بعد بی منفی...,5.580


In [6]:
import shutil
from google.colab import files
shutil.make_archive("mf3qa_analysis", "zip", "mf3qa_analysis")
files.download("mf3qa_analysis.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>